# 02 - Baseline Evaluation

Benchmarks the **untuned** Mistral 7B Instruct v0.3 on our function-calling test set. Establishes the "before" metrics that we'll compare against after fine-tuning.

**Key question**: How well does the base model handle enterprise function calling out of the box?

In [2]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes peft tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.8 MB/s eta 0:00:00


In [3]:
import json
import sys
import os

PROJECT_ROOT = "/kaggle/working/mistral-7b-enterprise-function-calling"  # <-- UPDATE THIS
sys.path.insert(0, PROJECT_ROOT)

from src.schemas import TOOL_SCHEMAS, SYSTEM_PROMPT
from src.utils import load_jsonl, dataset_stats
from src.inference import load_base_model, run_inference, run_inference_on_test_set
from src.evaluation import evaluate_results
from src.reporting import (
    overall_summary, breakdown_by_category, breakdown_by_tool,
    print_qualitative_examples,
)

In [4]:
DATA_DIR = f"{PROJECT_ROOT}/data"
test_data = load_jsonl(f"{DATA_DIR}/test.jsonl")

print(f"Test set size: {len(test_data)} examples")
dataset_stats(test_data)

Test set size: 154 examples
Total examples: 154

By category:
simple        62
complex       37
multi_tool    24
ambiguous     16
no_tool       15

Approx token lengths:
  Mean: 5019
  Min:  4937 | Max: 5204
  P95:  5100

✅ All examples well within 8192 token limit


In [5]:
model, tokenizer = load_base_model()

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [6]:
results = run_inference_on_test_set(model, tokenizer, test_data)


Running inference:   0%|          | 0/154 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Running inference: 100%|██████████| 154/154 [2:54:15<00:00, 67.90s/it]

Inference complete: 154 examples


In [7]:
# Before computing metrics, eyeball a few raw predictions to make sure
# inference is working correctly.
for i in range(3):
    print(f"\n{'='*60}")
    print(f"Example {i+1} | Category: {results[i]['category']}")
    print(f"{'='*60}")
    print(f"USER: {results[i]['input_messages'][1]['content'][:200]}...")
    print(f"\nEXPECTED: {results[i]['expected'][:200]}...")
    print(f"\nPREDICTED: {results[i]['predicted'][:200]}...")


Example 1 | Category: simple
USER: Can you run a search for institutional clients in the US? The query is 'BlackRock' and I need at most 50 results....

EXPECTED: {"name": "search_customers", "arguments": {"query": "BlackRock", "segment": "institutional", "country": "US", "max_results": 50}}...

PREDICTED: [{"name": "search_customers", "arguments": {"query": "BlackRock", "country": "US", "max_results": 50}}]...

Example 2 | Category: simple
USER: Please schedule a general meeting with client ACCT-7731 on May 5th 2025 for 90 minutes. The agenda is to discuss the upcoming contract renewal and new service tiers....

EXPECTED: {"name": "schedule_client_meeting", "arguments": {"client_id": "ACCT-7731", "meeting_type": "general", "preferred_date": "2025-05-05", "duration_minutes": 90, "agenda": "Discuss upcoming contract rene...

PREDICTED: [
  {
    "name": "schedule_client_meeting",
    "arguments": {
      "client_id": "ACCT-7731",
      "meeting_type": "general",
      "preferred_date":

In [8]:
eval_df = evaluate_results(results)
print(f"Evaluation complete: {len(eval_df)} examples scored")
eval_df.head(10)

Evaluation complete: 154 examples scored


,category,json_valid,func_name_correct,required_fields,type_correct,enum_compliant,hallucinated_params,argument_match,no_tool_restraint
0,simple,True,True,1.0,1.0,1.0,0.0,0.75,None
1,simple,True,True,1.0,1.0,1.0,0.0,0.80,None
2,simple,True,True,1.0,1.0,1.0,0.0,1.00,None
3,multi_tool,True,True,0.8,1.0,1.0,0.0,0.25,None
4,simple,True,True,1.0,1.0,1.0,0.0,1.00,None
5,simple,True,False,1.0,1.0,1.0,0.0,1.00,None
6,simple,True,True,1.0,1.0,1.0,0.0,1.00,None
7,ambiguous,True,True,1.0,1.0,1.0,0.0,0.60,None
8,simple,True,True,1.0,1.0,1.0,0.0,1.00,None
9,no_tool,None,None,NaN,NaN,NaN,NaN,NaN,True


In [9]:
summary = overall_summary(eval_df)
summary

,score
JSON Validity Rate,0.870504
Function Name Accuracy,0.917355
Required Fields Completeness,0.955372
Type Correctness Rate,1.000000
Enum Compliance Rate,1.000000
Avg Hallucinated Params,0.008264
Argument Value Match,0.768372
No-Tool Restraint Accuracy,1.000000


In [ ]:
# Where does the model struggle most? We expect:

cat_breakdown = breakdown_by_category(eval_df)
cat_breakdown

,json_valid,func_name_correct,required_fields,type_correct,enum_compliant,hallucinated_params,argument_match,no_tool_restraint,n
category,,,,,,,,,
ambiguous,0.8125,0.923077,0.923077,1.0,1.0,0.000000,0.714469,NaN,16
complex,0.837838,0.903226,0.961290,1.0,1.0,0.000000,0.652381,NaN,37
multi_tool,0.75,0.833333,0.877778,1.0,1.0,0.055556,0.542284,NaN,24
no_tool,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,15
simple,0.951613,0.949153,0.983051,1.0,1.0,0.000000,0.910169,NaN,62


In [ ]:
# Are some tools harder than others? Tools with nested objects/arrays

tool_breakdown = breakdown_by_tool(eval_df, results)
tool_breakdown

,json_valid,func_name_correct,required_fields,type_correct,enum_compliant,hallucinated_params,argument_match,n
tool_name,,,,,,,,
create_audit_task,1.0,1.0,1.000000,1.0,1.0,0.000000,0.654762,6
create_invoice,0.5,1.0,1.000000,1.0,1.0,0.000000,0.750000,8
escalate_issue,0.875,1.0,1.000000,1.0,1.0,0.000000,0.785714,8
extract_document_data,0.9,0.888889,0.851852,1.0,1.0,0.000000,0.685185,10
generate_report,0.75,0.833333,1.000000,1.0,1.0,0.000000,1.000000,8
get_customer_risk_profile,0.888889,0.625,1.000000,1.0,1.0,0.000000,0.875000,9
get_portfolio_summary,1.0,1.0,1.000000,1.0,1.0,0.000000,1.000000,2
get_transaction_history,1.0,1.0,1.000000,1.0,1.0,0.000000,0.894949,11
get_workflow_status,0.8,0.916667,0.900000,1.0,1.0,0.000000,0.761905,15


In [ ]:
# Save everything so we can compare against fine-tuned results in Notebook 04.

OUTPUT_DIR = f"{PROJECT_ROOT}/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save results as JSONL (one example per line, consistent with test.jsonl)
with open(f"{OUTPUT_DIR}/baseline_results.jsonl", "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

# Save eval_df as CSV
eval_df.to_csv(f"{OUTPUT_DIR}/baseline_eval_df.csv", index=True)

# Save summary as CSV
summary.to_csv(f"{OUTPUT_DIR}/baseline_summary.csv", index=True)

# Save cat_breakdown as CSV
cat_breakdown.to_csv(f"{OUTPUT_DIR}/baseline_cat_breakdown.csv", index=True)

# Save tool_breakdown as CSV
tool_breakdown.to_csv(f"{OUTPUT_DIR}/baseline_tool_breakdown.csv", index=True)

print(f"Results saved to {OUTPUT_DIR}/")

Results saved to /kaggle/working/mistral-7b-enterprise-function-calling/results/


In [ ]:
# The numbers tell part of the story, but seeing actual outputs is
# essential for understanding how the model fails.

print_qualitative_examples(results, eval_df, n_per_bucket=3)



######################################################################
# FAILURES
######################################################################

  Category: simple

  USER:
Hey, can you put together a client portfolio report for CUST-2209? Dates are 2024-02-01 to 2024-04-30. Skip the charts.

  EXPECTED:
{"name": "generate_report", "arguments": {"report_type": "client_portfolio", "entity_id": "CUST-2209", "period_start": "2024-02-01", "period_end": "2024-04-30", "include_charts": false}}

  PREDICTED:
[
  {
    "name": "get_portfolio_summary",
    "arguments": {
      "portfolio_id": "CUST-2209",
      "as_of_date": "2024-04-30",
      "include_benchmarks": false,
      "group_by": "asset_class"
    }
  },
  {
    "name": "generate_report",
    "arguments": {
      "report_type": "client_portfolio",
      "entity_id": "CUST-2209",
      "period_start": "2024-02-01",
      "period_end": "2024-04-30",
      "format": "html",
      "include_charts": false
    }
  }
]

  JSON=Tr

In [14]:
# Failure-mode breakdown: non-overlapping buckets so counts sum to total errors.

tool_examples = eval_df[eval_df["category"] != "no_tool"]
n_tool = len(tool_examples)

# Build a waterfall: each bucket removes examples claimed by previous ones
parseable = tool_examples[tool_examples["json_valid"] == True]
correct_func = parseable[parseable["func_name_correct"] == True]

failure_buckets = {
    "JSON Parse Failures": len(tool_examples) - len(parseable),
    "Wrong Function Name": len(parseable) - len(correct_func),
    "Missing Required Fields": (correct_func["required_fields"] < 1.0).sum() if len(correct_func) else 0,
    "Hallucinated Params": (correct_func["hallucinated_params"] > 0).sum() if len(correct_func) else 0,
}

print(f"Tool-call examples: {n_tool}\n")
for label, count in failure_buckets.items():
    print(f"  {label:.<30} {count:>4} / {n_tool}  ({count/n_tool:.1%})")

# No-tool restraint (separate population)
no_tool = eval_df[eval_df["category"] == "no_tool"]
if len(no_tool) > 0:
    false_triggers = (no_tool["no_tool_restraint"] == False).sum()
    print(f"\nNo-Tool False Triggers: {false_triggers} / {len(no_tool)} "
          f"({false_triggers/len(no_tool):.1%})")

Tool-call examples: 139

  JSON Parse Failures...........   18 / 139  (12.9%)
  Wrong Function Name...........   10 / 139  (7.2%)
  Missing Required Fields.......    6 / 139  (4.3%)
  Hallucinated Params...........    0 / 139  (0.0%)

No-Tool False Triggers: 0 / 15 (0.0%)


## Key Observations

### Overall Performance

| Metric | Score |
| --- | --- |
| JSON Validity Rate | 87.1% |
| Function Name Accuracy | 91.7% |
| Required Fields Completeness | 95.5% |
| Type Correctness | 100% |
| Enum Compliance | 100% |
| Argument Value Match | 76.8% |
| Hallucinated Params (avg) | 0.008 |
| No-Tool Restraint | 100% |

### Failure Mode Breakdown (139 tool-call examples)

1. **JSON Parse Failures**: 12.9% (18/139). This is the dominant failure mode. The base model sometimes produces malformed JSON, especially for tools with nested objects.
2. **Wrong Function Name**: 7.2% (10/139). The model occasionally confuses tools with overlapping semantics (e.g., `get_customer_risk_profile` vs `search_customers`).
3. **Missing Required Fields**: 4.3% (6/139). Relatively rare, meaning the model generally understands schema structure when it does produce valid JSON.
4. **Hallucinated Params**: 0.0% (0/139). The model never invents parameters outside the schema, which is a positive sign.
5. **No-Tool False Triggers**: 0.0% (0/15). The model shows perfect restraint on queries that should not trigger any function call.

### Category Difficulty (ranked by JSON validity)

| Category | JSON Valid | Func Name Correct | n |
| --- | --- | --- | --- |
| simple | 95.2% | 94.9% | 62 |
| complex | 83.8% | 90.3% | 37 |
| ambiguous | 81.3% | 92.3% | 16 |
| multi_tool | 75.0% | 83.3% | 24 |

As expected, **multi_tool** is the hardest category. The 7B model struggles to produce valid JSON when it needs to emit multiple tool calls in a single turn. Ambiguous and complex queries also degrade JSON formatting significantly compared to simple cases.

### Hardest Tools

- **create_invoice** (50% JSON validity): Likely due to its nested line-item arrays, which the model frequently malforms.
- **run_database_query** (57% JSON validity, 62.5% argument match): Free-form SQL strings inside JSON are difficult to quote correctly.
- **search_customers** (58.5% argument match): Many optional filter parameters lead to lower value accuracy.
- **get_customer_risk_profile** (62.5% function name accuracy): Confused with similar customer-related tools.

### Summary

The base Mistral 7B Instruct v0.3 performs reasonably well on simple, single-tool calls but degrades significantly on multi-tool and complex scenarios. The primary bottleneck is **JSON formatting** (not function selection or schema understanding). Fine-tuning should focus on:

- Reliable JSON output for nested/complex schemas
- Multi-tool call formatting
- Argument value precision for tools with many optional parameters

**These weaknesses are exactly what fine-tuning should address.**
